In [4]:
import pandas as pd
import requests

url = (
    "https://raw.githubusercontent.com/"
    "Vadimkin/ukrainian-air-raid-sirens-dataset/"
    "main/datasets/official_data_en.csv"
)

response = requests.get(url, timeout=60)
response.raise_for_status()

from io import BytesIO

df = pd.read_csv(BytesIO(response.content))

print("Розмір таблиці:", df.shape)
print("\nКолонки:")
print(df.columns.tolist())

print("\nПерші 5 рядків:")
display(df.head())

for col in ["oblast", "raion", "hromada", "level"]:
    if col in df.columns:
        print(f"\nЗначення {col}:")
        print(df[col].value_counts(dropna=False).head(15))
# kyiv_analysis.py
# Kyiv city vs Kyiv oblast: exploratory time-series analysis
# Data source: Vadimkin / ukrainian-air-raid-sirens-dataset
# All timestamps in the source are UTC.

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import requests

URL = (
    "https://raw.githubusercontent.com/"
    "Vadimkin/ukrainian-air-raid-sirens-dataset/"
    "main/datasets/official_data_en.csv"
)

OUT = Path("kyiv_results")
OUT.mkdir(exist_ok=True)

TEST_DAYS = 28
LOCAL_TZ = "Europe/Kyiv"

# 1. Download the latest available snapshot
response = requests.get(URL, timeout=60)
response.raise_for_status()

csv_path = OUT / "official_data_en.csv"
csv_path.write_bytes(response.content)

df = pd.read_csv(csv_path)

print("Columns:", df.columns.tolist())
print("Rows:", len(df))
print(df.head())

# 2. Identify required columns
required = {"started_at", "finished_at"}
missing = required - set(df.columns)

if missing:
    raise ValueError(
        f"Missing columns: {missing}. "
        f"Inspect the CSV schema before continuing."
    )

# 3. Parse timestamps (source uses UTC)
df["started_at"] = pd.to_datetime(
    df["started_at"], utc=True, errors="coerce"
)
df["finished_at"] = pd.to_datetime(
    df["finished_at"], utc=True, errors="coerce"
)

df = df.dropna(subset=["started_at", "finished_at"]).copy()

# Remove invalid intervals
df = df[df["finished_at"] > df["started_at"]].copy()

# Convert to Kyiv local time for daily and hourly patterns
df["start_local"] = df["started_at"].dt.tz_convert(LOCAL_TZ)
df["end_local"] = df["finished_at"].dt.tz_convert(LOCAL_TZ)

# 4. Inspect geographic fields and levels
geo_cols = [
    c for c in ["oblast", "raion", "hromada", "level"]
    if c in df.columns
]

print("\nGeographic fields:")
print(df[geo_cols].nunique(dropna=False))

for col in geo_cols:
    print(f"\n{col}:")
    print(df[col].value_counts(dropna=False).head(20))

# 5. Select Kyiv city and Kyiv oblast
# Do not assume the exact labels: inspect actual unique values first.
if "oblast" not in df.columns:
    raise ValueError(
        "No 'oblast' column. Adapt filtering to the actual schema."
    )

print("\nAvailable geographic labels:")
print(sorted(df["oblast"].dropna().astype(str).unique()))

# The city of Kyiv and Kyiv oblast must be separated.
# Adjust these patterns if the source uses different labels.
labels = df["oblast"].astype(str).str.strip().str.casefold()

city_mask = labels.isin({
    "kyiv city",
    "kyiv",
    "м. київ",
    "місто київ",
})

oblast_mask = labels.isin({
    "kyiv oblast",
    "kyivska oblast",
    "київська область",
})

kyiv = df[city_mask].copy()
oblast = df[oblast_mask].copy()

if kyiv.empty or oblast.empty:
    raise ValueError(
        "Could not identify both Kyiv city and Kyiv oblast. "
        "Check the printed labels and update the filters."
    )

print("\nKyiv city records:", len(kyiv))
print("Kyiv oblast records:", len(oblast))

# 6. Avoid counting district-level rows as separate whole-oblast alerts.
# This is a first-pass exploratory analysis. If the CSV contains
# multiple geographic levels, inspect 'level' before interpreting
# the counts as comparable numbers of alert events.

def daily_summary(data, area_name):
    data = data.copy()

    # Clip each alert to local calendar days so overnight alerts
    # contribute duration to every day they overlap.
    records = []

    for row in data.itertuples(index=False):
        start = row.start_local
        end = row.end_local

        first_day = start.normalize()
        last_day = end.normalize()
        day = first_day

        while day <= last_day:
            next_day = day + pd.DateOffset(days=1)
            segment_start = max(start, day)
            segment_end = min(end, next_day)

            minutes = max(
                0.0,
                (segment_end - segment_start).total_seconds() / 60
            )

            if minutes > 0:
                records.append({
                    "date": day.date(),
                    "area": area_name,
                    "alert_minutes": minutes,
                })

            day = next_day

    daily = pd.DataFrame(records)

    if daily.empty:
        return pd.DataFrame(
            columns=["date", "area", "alert_minutes"]
        )

    # Sum of alert intervals is not necessarily the same as
    # the total time a person was under at least one alert:
    # overlapping intervals can double-count minutes.
    return (
        daily.groupby(["date", "area"], as_index=False)
        ["alert_minutes"].sum()
    )

daily_city = daily_summary(kyiv, "Kyiv city")
daily_oblast = daily_summary(oblast, "Kyiv oblast")

daily = pd.concat(
    [daily_city, daily_oblast],
    ignore_index=True
)

daily["date"] = pd.to_datetime(daily["date"])
daily = daily.sort_values(["area", "date"])

daily.to_csv(OUT / "daily_alert_minutes.csv", index=False)

# 7. Create a complete daily calendar for each area
# Missing dates become zero only within the observed data window.
series = {}

for area_name, group in daily.groupby("area"):
    s = (
        group.set_index("date")["alert_minutes"]
        .sort_index()
    )

    calendar = pd.date_range(s.index.min(), s.index.max(), freq="D")
    series[area_name] = s.reindex(calendar, fill_value=0)

# 8. Summary statistics
summary_rows = []

for area_name, s in series.items():
    summary_rows.append({
        "area": area_name,
        "days": len(s),
        "total_alert_minutes": round(s.sum(), 1),
        "mean_daily_minutes": round(s.mean(), 1),
        "median_daily_minutes": round(s.median(), 1),
        "days_with_alerts": int((s > 0).sum()),
        "share_days_with_alerts_pct": round(
            (s > 0).mean() * 100, 2
        ),
    })

summary = pd.DataFrame(summary_rows)
summary.to_csv(OUT / "summary.csv", index=False)

print("\nSUMMARY")
print(summary.to_string(index=False))

# 9. Compare daily alert duration
fig, ax = plt.subplots(figsize=(12, 5))

for area_name, s in series.items():
    ax.plot(s.index, s.values / 60, label=area_name, linewidth=1)

ax.set_title("Daily recorded air-alert duration")
ax.set_xlabel("Date")
ax.set_ylabel("Alert-hours per day")
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(OUT / "daily_comparison.png", dpi=160)
plt.close(fig)

# 10. Hour-of-day pattern
hourly_rows = []

for area_name, data in [
    ("Kyiv city", kyiv),
    ("Kyiv oblast", oblast),
]:
    counts = (
        data["start_local"].dt.hour
        .value_counts()
        .reindex(range(24), fill_value=0)
        .sort_index()
    )

    for hour, count in counts.items():
        hourly_rows.append({
            "area": area_name,
            "hour": hour,
            "alert_starts": int(count),
        })

hourly = pd.DataFrame(hourly_rows)
hourly.to_csv(OUT / "hourly_start_counts.csv", index=False)

fig, ax = plt.subplots(figsize=(10, 5))

for area_name, group in hourly.groupby("area"):
    ax.plot(group["hour"], group["alert_starts"],
            marker="o", label=area_name)

ax.set_xticks(range(24))
ax.set_xlabel("Hour of day (Kyiv local time)")
ax.set_ylabel("Recorded alert starts")
ax.set_title("Alert starts by hour")
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(OUT / "hourly_patterns.png", dpi=160)
plt.close(fig)

# 11. Simple leakage-safe baseline forecast:
# predict each day using the mean of the previous 7 days.
# Evaluate only the latest TEST_DAYS of the common observed period.

common_start = max(s.index.min() for s in series.values())
common_end = min(s.index.max() for s in series.values())

forecast_rows = []

for area_name, s in series.items():
    s = s.loc[common_start:common_end].copy()

    if len(s) <= TEST_DAYS + 7:
        print(f"Not enough data for forecast evaluation: {area_name}")
        continue

    test_start = s.index[-TEST_DAYS]

    for date in s.index[s.index >= test_start]:
        history = s.loc[s.index < date].tail(7)

        if len(history) < 7:
            continue

        forecast_rows.append({
            "area": area_name,
            "date": date,
            "actual_minutes": float(s.loc[date]),
            "predicted_minutes": float(history.mean()),
        })

forecast = pd.DataFrame(forecast_rows)

if not forecast.empty:
    forecast["absolute_error"] = (
        forecast["actual_minutes"]
        - forecast["predicted_minutes"]
    ).abs()

    metrics = (
        forecast.groupby("area")
        .agg(
            MAE_minutes=("absolute_error", "mean"),
            test_days=("date", "count"),
        )
        .reset_index()
    )

    metrics.to_csv(OUT / "forecast_metrics.csv", index=False)
    forecast.to_csv(OUT / "forecast_test_predictions.csv", index=False)

    print("\nFORECAST METRICS")
    print(metrics.to_string(index=False))

print(f"\nResults saved in: {OUT.resolve()}")


Розмір таблиці: (299275, 7)

Колонки:
['oblast', 'raion', 'hromada', 'level', 'started_at', 'finished_at', 'source']

Перші 5 рядків:


,oblast,raion,hromada,level,started_at,finished_at,source
0,Vinnytska oblast,NaN,NaN,oblast,2022-03-15 16:10:34+00:00,2022-03-15 16:50:07+00:00,official
1,Zhytomyrska oblast,NaN,NaN,oblast,2022-03-15 16:11:25+00:00,2022-03-15 16:54:23+00:00,official
2,Cherkaska oblast,Umanskyi raion,Umanska terytorialna hromada,hromada,2022-03-15 16:11:50+00:00,2022-03-15 16:54:47+00:00,official
3,Mykolaivska oblast,Pervomaiskyi raion,m. Pervomaisk ta Pervomaiska terytorialna hromada,hromada,2022-03-15 16:14:46+00:00,2022-03-15 16:57:08+00:00,official
4,Kirovohradska oblast,NaN,NaN,oblast,2022-03-15 16:15:11+00:00,2022-03-15 16:54:52+00:00,official



Значення oblast:
oblast
Dnipropetrovska oblast    45729
Kharkivska oblast         39651
Donetska oblast           28117
Zaporizka oblast          26744
Sumska oblast             21036
Poltavska oblast          15988
Mykolaivska oblast        15699
Chernihivska oblast       14899
Odeska oblast             13238
Khersonska oblast         12648
Kirovohradska oblast      11743
Kyivska oblast            10951
Cherkaska oblast          10370
Vinnytska oblast           5021
Zhytomyrska oblast         4711
Name: count, dtype: int64

Значення raion:
raion
NaN                     130235
Nikopolskyi raion        19800
Kharkivskyi raion         9281
Pokrovskyi raion          6134
Zaporizkyi raion          5500
Chuhuivskyi raion         3230
Kryvorizkyi raion         3152
Sumskyi raion             3129
Kupianskyi raion          3113
Bohodukhivskyi raion      3104
Iziumskyi raion           3084
Synelnykivskyi raion      2768
Konotopskyi raion         2716
Pavlohradskyi raion       2654
Lozivskyi ra